# Design

## Scope and source evidence

The three assigned fits are the OLS time trends underlying Figure 1: avoiding quarterly losses, earnings decreases, and negative earnings surprises. The annual regression sample is 1985–2002 (18 years); the paper separately reports a 111,485 firm-quarter main sample. I use calendar year of the I/B/E/S forecast-period end date (`PENDS`/`FPEDATS`) for the annual grouping, retaining 1984 actuals to form the four-quarter lag for 1985. This follows the stated 1984–2002 actual-data window and the lagged-quarter explanation (paper pp. 427–428; printed pp. 427–428; Figure 1 on p. 430).

Paper p. 427 states that the sample needs actual EPS in quarter t and t−4, an analyst forecast, a beginning-of-quarter price, and three days of returns around the actual announcement. It excludes CRSP SIC 4400–5000 and 6000–6500 and trims the two extreme CAR percentiles; the reported main sample is 111,485 firm-quarters. Footnotes 8–10 specify the latest pre-announcement forecast and same-day mean, the primary beginning-of-quarter-t deflator for all three earnings measures, and that the extreme-percentile CAR deletion is an outlier screen. Paper p. 428 defines the standardized difference as the actual minus expected bin count divided by its estimated standard deviation, with expected count equal to the average of adjacent bins (footnote 12). Footnote 15 standardizes each year/threshold to N=1,000 using that year’s bin proportions. Footnote 13 includes zero in the threshold-achieving/right-of-zero bin. Figure 1 uses .0025 bins and the bin just left of zero; slopes are OLS on year (paper pp. 428–430; Figure 1 p. 430).

The paper PDF image was inspected for physical page 5 (sample selection and forecast footnote 8) and physical pages 6–8 because extracted text clipped mathematical signs and the Figure 1 labels. Footnote 12 gives

`Var(D_i) ≈ N p_i(1-p_i) + (1/4)N(p_{i-1}+p_{i+1})(1-p_{i-1}-p_{i+1})`,

where `D_i` is the observed count less the mean of the two adjacent counts. Figure 1 reports slopes/t-statistics of −0.0361/−1.63, −0.1024/−1.62, and −1.218/−13.91 and adjusted R² of 9%, 9%, and 92%, respectively (p. 430). These are comparison targets, not construction inputs.

## Data mapping and construction

The executed build uses `IBES.ACTU_EPSUS` and `IBES.DETU_EPSUS`; the local SAS catalog labels them “Actuals Unadjusted” and “Detail History – Unadjusted.” This is an explicit deviation from the paper’s main data basis: physical PDF p. 15 (printed p. 437, “Split-Adjusted Data”) describes split-adjusted analyses and reports unadjusting with I/B/E/S factors as a sensitivity analysis. The current CRSP-factor construction is therefore an unadjusted, sensitivity-style approximation, not a neutral source correction, not the paper’s exact unadjusting sensitivity, and not an exact primary reproduction. It removes the rounding behavior the paper discusses.

I checked the full frozen I/B/E/S catalog (`output/feasibility_runs/resources/catalog.json` and the corresponding catalog sections in `/storage/sas/data_description.txt`). The adjusted `ACT_EPSUS` and `DET_EPSUS` products are present; the 2009 Detail History guide says Detail values are adjusted and that unadjusted files are separately available (physical PDF pp. 18–19). Their cataloged fields are earnings and dates, not a historical stock-price series. The only catalog variables labeled as prices are price-target variables. `HDXRATI` and `HSXRAT` are exchange-rate files. The catalog also has `SPLIT_GUIDANCE` under the I/B/E/S Guidance sublibrary, with event dates and a split-factor field; its documentation does not establish that it is the Detail History Adjustments file or define a cumulative factor basis that can be matched to the needed price. The 2009 guide describes a separate monthly Detail “Adjustments” file containing cumulative split factors (physical PDF p. 16), but that file is not in the frozen catalog. The paper’s stock-price series and the matching split-factor history are therefore unavailable here. Using adjusted `ACT_EPSUS`/`DET_EPSUS` with a CRSP price would mix undocumented terminal share bases; I do not do that or infer factors from the guidance product.

For the executable approximation, I use the unadjusted I/B/E/S products with date-specific *relative* CRSP `CFACSHR` ratios and the CRSP price. This is internally on the CRSP share basis at the quarter-opening price date. It follows the documented logic in the [WRDS provider note *A Note on IBES Unadjusted Data*](https://wrds-www.wharton.upenn.edu/documents/5/A_Note_on_IBES_Unadjusted_Data_pdf.pdf) (2006, pp. 5–6), which aligns I/B/E/S unadjusted per-share values with CRSP factors at estimate and report dates; applying the ratio to the quarter-opening price date is my explicit extension. This implementation is best-effort and cannot be read as source-faithful to the paper’s split-adjusted primary or its I/B/E/S-factor sensitivity.

“Unadjusted” refers to share/split adjustment, not absence of currency conversion. Actuals use `MEASURE='EPS'`, `PDICITY='QTR'`, `USFIRM=1`, and company-level `CURR_ACT='USD'`. Forecasts use quarterly `FPI` codes `6`–`9`, `N`–`Q`, `R`–`T`, `L`, and `Y`. The 2009 Detail History guide (physical PDF p. 41) lists quarters 1–8 and Y for more than eight quarters ahead; the [2025 WRDS LSEG IBES query guide](https://www.lib.nccu.edu.tw/var/file/0/1000/img/101/WRDS_IBES_202511.pdf) identifies R/S/T/L as quarters 9–12 and Y as more than 12 quarters ahead. Forecasts also use `MEASURE='EPS'`, `USFIRM=1`, and `REPORT_CURR='USD'`, the stored-value currency basis. The raw forecast fields `CURR`, estimate-level `CURRFL`, and `REPORT_CURR` are retained in the scratch audit and cross-tabbed before the screen. `CURR` is blank in the local U.S. product; `CURRFL='C'` indicates an estimate received in Canadian dollars, while a blank flag indicates it was received in U.S. dollars. The Detail History guide (2009, physical PDF pp. 18–19) states that Detail values are converted to the company-level report currency and that the estimate-level flag records the received basis. I therefore select stored forecast values with `REPORT_CURR='USD'` to align them with USD CRSP prices, and retain both blank and `C` `CURRFL` records when the stored report currency is USD. I do not apply an annual FX factor or infer stored-value currency from the received-currency flag. Zero EPS values remain valid; missing values remain missing. If repeated ticker-period actual rows occur, the earliest public `ANNDATS`, then earliest `ACTDATS`/`ACTTIMS` row on that date, is retained after logging duplicate counts and value/date conflicts. The year −4 actual is matched on I/B/E/S ticker and the closest period end within seven calendar days of the same fiscal quarter one year earlier; nonunique or absent lag matches are left missing and excluded from the complete sample.

The fields and labels are verified in the local SAS catalog (`/storage/sas/data_description.txt`): IBES `PENDS`, `FPEDATS`, `ANNDATS`, `ACTDATS`, `VALUE`, currency, periodicity, measure, and `USFIRM`; CRSP `DSENAMES` `NCUSIP`, `NAMEDT`, `NAMEENDT`, `SICCD`; CRSP `DSF` `PERMNO`, `DATE`, `PRC`, `RET`, `CFACSHR`; and CRSP `DSI` `DATE`, `VWRETD`. I link an IBES actual CUSIP to the historical CRSP `NCUSIP` valid on the actual announcement date, requiring a unique effective-dated PERMNO match. CRSP’s cached Stock and Index Data Descriptions guide (Chapter 2, printed pp. 23–24 and name-history description in Chapter 6, printed pp. 120–121) distinguishes effective-dated name-history CUSIPs from header identifiers. CRSP’s adjustment formula (Chapter 3, printed p. 94) defines adjusted shares as raw shares times the cumulative share factor. The WRDS provider note *A Note on IBES Unadjusted Data* (2006, pp. 5–6) documents matching IBES unadjusted per-share values to CRSP factors at dated estimate/report records; I extend that dated ratio to the paper’s quarter-opening price basis.

For each fiscal quarter t, the quarter start is the first day after the prior three-month period ends: `INTNX('month',PENDS,-3,'end')+1`; for an off-calendar `PENDS` of 31 July, this gives 1 May. The price proxy is the first available CRSP close with nonzero absolute `PRC` from that date through ten calendar days later. This finite search window is explicit in code; the build reports the selected date-age distribution and does not screen on price age. Negative CRSP `PRC` values, which encode bid/ask averages, are converted to absolute price. In the executed approximation, each EPS or forecast value measured at source date s is placed on the quarter-opening CRSP share basis q as `value_s × CFACSHR_q / CFACSHR_s`. The t−4 actual is converted to the same q basis before differencing. The threshold inputs are then `EPS_t/price_q`, `(EPS_t−EPS_{t−4})/price_q`, and `(EPS_t−forecast_t)/price_q`. Actuals use the local `ANNDATS` field as the provider-labeled announcement date; the build logs its coverage and its differences from `ACTDATS`. The guide notes that pre-1999 actual-date coverage may reflect activation dates (2009, physical PDF p. 16), so those event dates are treated as the best available source proxy. Both price and numerators are USD-denominated in this construction, but this does not establish that they share the paper’s precise I/B/E/S split-adjusted price basis. The ratio uses relative CRSP factors only and assumes no cross-provider cumulative-factor terminal base.

For the surprise threshold, I retain all available detailed estimates with announcement dates strictly before the actual announcement; the paper gives no maximum lookback, so I impose no numerical forecast-age cap. I select the latest such date for the matched TICKER/FPEDATS and average the analyst estimates on that date. Repeated same-analyst/date records are checked before selecting one dated estimate per analyst. The actual announcement date is the event date. I map it to the first CRSP market date on or after the announcement date and sum daily `RET−VWRETD` for trading days −1, 0, and +1. SIC is taken from the effective CRSP name-history row at the event date. I apply the two SIC exclusions, then the 1st/99th percentile CAR screen. The market proxy is CRSP’s value-weighted daily return including distributions.

I retain a construction-stage attrition log that separates absent quarter rows from missing fields on observed rows, reports ambiguous links and duplicate/vintage groups, reports day gaps for unique t−4 period matches, and reports price age from quarter start without applying another price-age filter. The forecast currency cross-tab covers 6,382,523 in-period Detail rows: `CURR` is blank throughout; with blank `CURRFL` versus `C`, `REPORT_CURR` is USD for 6,324,497 versus 15,382 rows, CAD for 10,720 versus 28,323, JPY for 1 versus 0, and missing for 3,592 versus 8. This confirms why `REPORT_CURR` screens the stored value currency while `CURRFL` is retained as received-currency provenance; the selected latest-date forecasts include both blank and C flags on USD stored values. Actual currency screens retain 456,551 rows, with USD in 99.47%; duplicates at TICKER/PENDS are zero. The latest complete sample spans PENDS 31JAN1985–31DEC2002, ANNDATS 11FEB1985–28AUG2003, and quarter-opening price dates 01NOV1984–01OCT2002, confirming the required 2003 event-window extension. The final handoff is one row per year with each threshold’s N and standardized difference; Stata therefore estimates three 18-observation trend regressions from `data/reg.dta`.

## Estimation and assumptions

Each fit is `regress` of the threshold-year standardized difference on calendar year, with the default intercept and conventional homoskedastic OLS standard errors. The paper reports t-statistics but does not specify a covariance estimator; conventional OLS inference is a best-effort choice for the 18 annual observations. Stata/`esttab` reports two-sided p-values, while Figure 1’s threshold-criterion discussion uses one-tailed language. A supporting pooled interaction restriction compares the three time slopes as the paper’s reported Chow test; this does not replace any of the three assigned fits.

The exact IBES–CRSP link, CRSP price proxy for the paper’s named IBES price, quarter-start date convention, fiscal-period-to-calendar-year grouping, event mapping when announcements occur on nontrading days, and CRSP’s market-adjusted return implementation are not fully described in the paper. The provider warns that actual dates before January 1999 can reflect activation dates, which limits announcement-event precision for the early years. The choices above are best-effort and are reported as such. The source’s quarter-year labels define the sample period, so CRSP event-window dates are allowed to extend into 2003 when 2002 fiscal-quarter announcements fall there. No analyst-following-three screen is imposed; p. 427 limits that screen to forecast-dispersion analysis, outside the assigned fits.

## Execution results and comparison

SAS retained 333,156 current actuals for 1985–2002. The t−4 actual match is unique for 284,934; 283,757 matches have the exact same quarter-end date and 1,177 are one calendar day away under the seven-day nearest-date rule. Another 48,222 have no unique lag match. Of the current actuals, 229,688 have a timely forecast, 318,099 have a beginning-quarter price, 321,290 have a unique effective-dated PERMNO/SIC match, and 320,001 have complete three-day CAR. The corrected link audit finds 11,866 without a PERMNO match, no missing SIC among the unique matches, and no ambiguous effective links. After all three threshold inputs, SIC exclusions, and CAR completeness, 163,938 remain before trimming; 1,640 low and 1,639 high CAR observations are deleted, leaving 160,659. This is 49,174 (44.1%) above the paper’s 111,485. The final sample has no duplicate PERMNO/PENDS keys. The available diagnostics do not isolate a single source of this difference; the paper’s IBES price field and market-adjusted return implementation, effective link choices, actual-date coverage, and source vintage remain unresolved rather than being attributed to vintage alone. Executed annual firm-quarter N averages 8,925.5 (range 3,836–13,736), compared with the paper’s reported threshold-specific annual means of 6,114–6,161 in footnote 15.

The selected price is the first available nonzero CRSP close within ten calendar days of quarter start. Among all 333,156 current actuals, 318,111 have such a price: mean age 0.75 calendar days, median 0, 90th percentile 2, 95th and 99th percentiles 3, maximum 10. In the final sample, mean age is 0.72 days; 55.48% use a same-day close, 94.70% are priced within two days, and only one observation uses a day-10 close. The build reports these ages but does not apply a separate price-age restriction. In the final firm-quarter sample the means are 0.2607 for current EPS, 0.2509 for lagged EPS, 0.2905 for mean forecast, 0.0011 for loss, −0.00045 for the earnings change, −0.00651 for the surprise, 0.00246 for three-day CAR, and $22.42 for price. The annual firm-quarter count ranges from 3,836 in 1985 to 13,736 in 1998.

All three OLS fits use 18 annual observations. Results below compare the exported table and Stata log with Figure 1 (paper p. 430); the estimates were not tuned to the paper.

| Trend | Slope (SE) | t | Two-sided p | Adjusted R² | Paper slope / t / Adj. R² |
|---|---:|---:|---:|---:|---:|
| Avoid losses | −0.0344 (0.0304) | −1.13 | 0.274 | 0.016 | −0.0361 / −1.63 / 0.09 |
| Avoid earnings decreases | −0.0960 (0.0515) | −1.86 | 0.081 | 0.127 | −0.1024 / −1.62 / 0.09 |
| Avoid negative surprises | −0.9758 (0.0721) | −13.54 | <0.001 | 0.915 | −1.218 / −13.91 / 0.92 |

The pooled slope-difference test rejects equality of the decrease and surprise slopes relative to the loss slope, `F(2,48)=94.86`, `p<0.001`, consistent with the paper’s 1% Chow-test conclusion. Descriptive rows and all fitted sample sizes are in `tables/figure1_annual_descriptives.*` and `tables/figure1_trend_ols.*`; the generated trend figure is `figures/figure1_thresholds.pdf` and `.png`. The exported tables report two-sided p-values per template; the paper describes one-tailed threshold evidence. The main unresolved empirical discrepancy is sample construction, which also shifts the annual standardized differences and the three slopes.


In [ ]:
options errorabend mprint msglevel=i;
libname ibes '/storage/sas/base/ibes' access=readonly;
libname crsp '/storage/sas/base/crsp/a_stock' access=readonly;
libname out 'data';
libname aud 'scratch';

title 'W2028237122: quarterly threshold time trends';

/* Retain the full t-4 actual history before restricting the trend sample. */
data work.actraw;
    set ibes.actu_epsus(keep=ticker cusip pends measure pdicity anndats anntims actdats acttims value curr_act usfirm
        where=(pends >= '01JAN1984'd and pends < '01JAN2003'd));
run;

title 'IBES actual field and code audit';
proc freq data=work.actraw;
    tables measure pdicity usfirm curr_act / missing;
run;

data work.actusd;
    set work.actraw;
    if measure ne 'EPS' or pdicity ne 'QTR' or usfirm ne 1 or curr_act ne 'USD' then delete;
run;

proc sql;
    create table work.actdatecheck as
    select year(pends) as period_year,
           count(*) as n_actual_rows,
           sum(missing(anndats)) as n_missing_announce_date,
           sum(missing(actdats)) as n_missing_activation_date,
           sum(not missing(anndats) and not missing(actdats) and anndats ne actdats)
               as n_act_anndate_diff,
           min(anndats) as first_announce_date format=date9.,
           max(anndats) as last_announce_date format=date9.
    from work.actusd
    group by calculated period_year;
quit;
title 'Actual announcement and activation date coverage by fiscal-period year';
proc print data=work.actdatecheck noobs; run;

proc sql;
    create table work.actmiss as
    select count(*) as n_period_rows,
           sum(missing(value)) as n_missing_value,
           sum(value=0) as n_zero_value,
           sum(missing(anndats)) as n_missing_announce,
           sum(missing(cusip)) as n_missing_cusip
    from work.actusd;
    create table work.actdup as
    select ticker, pends, count(*) as nrows,
           count(distinct value) as nvalues,
           count(distinct anndats) as nann
    from work.actusd
    where not missing(value) and not missing(anndats) and not missing(cusip)
    group by ticker, pends
    having count(*) > 1;
    create table work.actdupsum as
    select count(*) as duplicate_ticker_periods,
           sum(nrows-1) as extra_rows,
           sum(nvalues>1) as periods_with_value_conflicts,
           sum(nann>1) as periods_event_date_conflicts
    from work.actdup;
quit;

title 'IBES actual rows and duplicate diagnostics';
proc print data=work.actmiss noobs; run;
proc print data=work.actdupsum noobs; run;

data work.actgood;
    set work.actusd;
    if missing(value) or missing(anndats) or missing(cusip) then delete;
run;
proc sort data=work.actgood;
    by ticker pends anndats actdats acttims;
run;
data work.actuni;
    set work.actgood;
    by ticker pends;
    if first.pends;
run;

data work.qcurr;
    set work.actuni;
    if year(pends) < 1985 or year(pends) > 2002 then delete;
    qyear=year(pends);
    qbegin=intnx('month',pends,-3,'end')+1;
    lagtarget=intnx('year',pends,-1,'same');
    rowid=_n_;
    format pends anndats actdats qbegin lagtarget date9.;
run;

/* Exact TICKER period matching is not assumed: use the closest t-4 period
   within one week of the same fiscal-quarter date, requiring a unique nearest row. */
proc sql;
    create table work.lagcand as
    select a.rowid, b.pends as lagpends, b.anndats as lagdate,
           b.value as lagraw, abs(b.pends-a.lagtarget) as daygap
    from work.qcurr as a inner join work.actuni as b
      on a.ticker=b.ticker
     and b.pends >= a.lagtarget-7
     and b.pends <= a.lagtarget+7;
    create table work.lagmind as
    select rowid, min(daygap) as minday
    from work.lagcand group by rowid;
    create table work.lagbest as
    select a.rowid, a.lagpends, a.lagdate, a.lagraw, a.daygap
    from work.lagcand as a inner join work.lagmind as b
      on a.rowid=b.rowid and a.daygap=b.minday;
    create table work.lagagg as
    select rowid, count(*) as nlagbest,
           max(lagpends) as lagpends,
           max(lagdate) as lagdate,
           max(lagraw) as lagraw,
           max(daygap) as laggap
    from work.lagbest group by rowid;
quit;

/* The historical CRSP name row is selected using its effective CUSIP date range. */
proc sql;
    create table work.linkraw as
    select a.rowid, b.permno, b.siccd, b.shrcd
    from work.qcurr as a left join crsp.dsenames as b
      on strip(a.cusip)=strip(b.ncusip)
     and b.namedt <= a.anndats
     and (missing(b.nameendt) or b.nameendt >= a.anndats);
    create table work.linkagg as
    select rowid, count(distinct permno) as nperm,
           count(distinct siccd) as nsic,
           min(permno) as permno,
           min(siccd) as siccd,
           min(shrcd) as shrcd
    from work.linkraw group by rowid;
    create table work.lagcheck as
    select count(*) as nbase,
           sum(missing(b.rowid)) as n_no_lag,
           sum(b.nlagbest=1) as n_unique_lag,
           sum(b.nlagbest>1) as n_ambiguous_lag
    from work.qcurr as a left join work.lagagg as b on a.rowid=b.rowid;
    create table work.linkcheck as
    select count(*) as nbase,
           sum(missing(b.nperm) or b.nperm=0) as n_no_permno_match,
           sum(b.nperm=1 and b.nsic=0) as n_missing_sic,
           sum(b.nperm=1 and b.nsic=1) as n_unique_effective_links,
           sum(b.nperm>1 or b.nsic>1) as n_ambiguous_effective_links
    from work.qcurr as a left join work.linkagg as b on a.rowid=b.rowid;
quit;

title 'Lag and effective-dated IBES–CRSP link diagnostics';
proc print data=work.lagcheck noobs; run;
proc print data=work.linkcheck noobs; run;
title 'Date gaps for unique t-4 actual matches';
proc freq data=work.lagagg;
    where nlagbest=1;
    tables laggap;
run;

/* Keep every candidate row for staged attrition; ambiguous matches stay missing. */
proc sql;
    create table work.core0 as
    select a.*, b.lagpends, b.lagdate, b.lagraw, b.nlagbest,
           c.permno, c.siccd, c.shrcd, c.nperm, c.nsic
    from work.qcurr as a
    left join work.lagagg as b on a.rowid=b.rowid
    left join work.linkagg as c on a.rowid=c.rowid;
quit;
data work.core;
    set work.core0;
    if nlagbest ne 1 then do; lagpends=.; lagdate=.; lagraw=.; end;
    if nperm ne 1 or nsic ne 1 then do; permno=.; siccd=.; end;
    format lagpends lagdate date9.;
run;

/* Validate the quarterly EPS detail mapping before screening forecast currencies. */
data work.fcraw;
    set ibes.detu_epsus(keep=ticker cusip fpedats fpi measure usfirm analys estimator
        anndats anntims actdats acttims revdats revtims value curr currfl report_curr
        where=(fpedats >= '01JAN1985'd and fpedats < '01JAN2003'd));
run;
title 'IBES detailed estimate field and currency audit';
proc freq data=work.fcraw;
    tables measure usfirm fpi curr*currfl*report_curr / missing;
run;
proc freq data=work.fcraw noprint;
    tables curr*currfl*report_curr / missing out=aud.forecast_currency;
run;
title 'Raw estimate-level and company-level currency cross-tab';
proc print data=aud.forecast_currency noobs; run;

data work.fcusd;
    set work.fcraw;
    if measure ne 'EPS' or usfirm ne 1 then delete;
    if fpi not in ('6','7','8','9','N','O','P','Q','R','S','T','L','Y') then delete;
    /* DETU values are split-unadjusted but already on the report-currency basis.
       CURR/CURRFL retain the estimate-level received-currency provenance. */
    if report_curr ne 'USD' then delete;
    if missing(value) or missing(anndats) or missing(analys) then delete;
run;

/* Latest public forecast date strictly before the actual announcement; same-day
   estimates are averaged after retaining one time-stamped row per analyst. */
proc sql;
    create table work.fccand as
    select a.rowid, f.ticker, f.fpedats, f.analys, f.estimator,
           f.fpi, f.anndats, f.anntims, f.actdats, f.acttims,
           f.revdats, f.revtims, f.value, f.curr, f.currfl, f.report_curr
    from work.core as a inner join work.fcusd as f
      on a.ticker=f.ticker and a.pends=f.fpedats
     and f.anndats < a.anndats;
    create table work.fccandcheck as
    select count(*) as forecast_candidates,
           count(distinct rowid) as firm_quarters_with_a_forecast,
           sum(missing(value)) as missing_candidate_values
    from work.fccand;
quit;

proc sort data=work.fccand;
    by rowid descending anndats descending anntims;
run;
data work.fdate;
    set work.fccand;
    by rowid;
    if first.rowid;
    fdate=anndats;
    keep rowid fdate;
    format fdate date9.;
run;
proc sql;
    create table work.fclatest0 as
    select a.*
    from work.fccand as a inner join work.fdate as b
      on a.rowid=b.rowid and a.anndats=b.fdate;
    create table work.fcdup as
    select rowid, analys, count(*) as nrows,
           count(distinct value) as nvalues,
           count(distinct anntims) as ntimes
    from work.fclatest0
    group by rowid, analys
    having count(*) > 1;
    create table work.fcdupsum as
    select count(*) as analyst_date_groups_with_repeats,
           sum(nrows-1) as extra_rows,
           sum(nvalues>1) as groups_with_value_conflicts,
           sum(ntimes>1) as groups_with_time_conflicts
    from work.fcdup;
quit;
proc sort data=work.fclatest0;
    by rowid analys descending anntims descending actdats descending acttims
       descending revdats descending revtims;
run;
data work.fcperanal;
    set work.fclatest0;
    by rowid analys;
    if first.analys;
run;
data aud.forecast_selected;
    set work.fcperanal;
run;
proc sql;
    create table work.fcast as
    select rowid, mean(value) as forecast,
           count(distinct analys) as analyst_n,
           max(anndats) as fdate
    from work.fcperanal group by rowid;
    create table work.fcsummary as
    select count(*) as n_final_forecast_rows,
           mean(analyst_n) as mean_analysts_on_selected_day,
           min(fdate) as earliest_selected_forecast,
           max(fdate) as latest_selected_forecast
    from work.fcast;
quit;
title 'Forecast date, duplicate, and selected-forecast diagnostics';
proc print data=work.fccandcheck noobs; run;
proc print data=work.fcdupsum noobs; run;
proc print data=work.fcsummary noobs; run;

/* CRSP market calendar for event dates; fiscal 2002 observations may announce in 2003. */
data work.cal;
    set crsp.dsi(keep=date vwretd where=(date >= '01JAN1983'd and date < '01JAN2004'd));
    if missing(vwretd) then delete;
run;
proc sort data=work.cal; by date; run;
data work.cal;
    set work.cal;
    tradeidx=_n_;
run;

proc sql;
    create table work.eventdate0 as
    select a.rowid, min(c.date) as evdate
    from work.core as a left join work.cal as c
      on c.date >= a.anndats and c.date <= a.anndats+7
    group by a.rowid;
    create table work.eventdate as
    select a.rowid, c.tradeidx as eidx, c.date as evtrade
    from work.eventdate0 as a left join work.cal as c on a.evdate=c.date;
    create table work.pdate0 as
    select a.rowid, min(d.date) as pdate
    from work.core as a left join crsp.dsf as d
      on a.permno=d.permno
     and d.date >= a.qbegin and d.date <= a.qbegin+10
     and abs(d.prc) > 0
    group by a.rowid;
    create table work.pdata as
    select a.rowid, a.pdate, abs(d.prc) as price,
           d.cfacshr as cfac_p
    from work.pdate0 as a inner join work.core as c on a.rowid=c.rowid
    left join crsp.dsf as d
      on c.permno=d.permno and a.pdate=d.date;
quit;

/* Resolve source share factors at the last security trading day on/before the
   actual, t-4 actual, and selected analyst forecast dates. */
data work.facreqa;
    set work.core;
    length role $1;
    if not missing(permno) and not missing(anndats) then do;
        role='A'; reqdate=anndats; output;
    end;
    if not missing(permno) and nlagbest=1 and not missing(lagdate) then do;
        role='L'; reqdate=lagdate; output;
    end;
    format reqdate date9.;
    keep rowid permno role reqdate;
run;
proc sql;
    create table work.facreqf as
    select a.rowid, a.permno, 'F' as role length=1,
           b.fdate as reqdate
    from work.core as a inner join work.fcast as b on a.rowid=b.rowid
    where not missing(a.permno) and not missing(b.fdate);
quit;
data work.facreq2;
    set work.facreqa work.facreqf;
    if missing(permno) then delete;
    format reqdate date9.;
run;

proc sql;
    create table work.facdate as
    select r.rowid, r.role, max(d.date) as fdate
    from work.facreq2 as r left join crsp.dsf as d
      on r.permno=d.permno
     and d.date <= r.reqdate and d.date >= r.reqdate-7
    group by r.rowid, r.role;
    create table work.factors0 as
    select a.rowid, a.role, a.fdate, d.cfacshr as cfac
    from work.facdate as a inner join work.core as c on a.rowid=c.rowid
    left join crsp.dsf as d
      on c.permno=d.permno and d.date=a.fdate;
    create table work.factors as
    select rowid,
           max(case when role='A' then cfac else . end) as cfac_a,
           max(case when role='L' then cfac else . end) as cfac_l,
           max(case when role='F' then cfac else . end) as cfac_f,
           max(case when role='A' then fdate else . end) as fdate_a,
           max(case when role='L' then fdate else . end) as fdate_l,
           max(case when role='F' then fdate else . end) as fdate_f
    from work.factors0 group by rowid;
quit;

/* Build the three-day market-adjusted CAR from trading days -1, 0, +1. */
proc sort data=work.core; by rowid; run;
proc sort data=work.eventdate; by rowid; run;
data work.evtreq;
    merge work.core(in=a keep=rowid permno) work.eventdate(in=b keep=rowid eidx);
    by rowid;
    if a and not missing(permno) and not missing(eidx) then do;
        offset=-1; wanted=eidx-1; output;
        offset=0; wanted=eidx; output;
        offset=1; wanted=eidx+1; output;
    end;
    keep rowid permno offset wanted;
run;
proc sql;
    create table work.evtdays0 as
    select a.rowid, a.permno, a.offset, b.date, b.vwretd
    from work.evtreq as a inner join work.cal as b on a.wanted=b.tradeidx;
    create table work.evtdays as
    select a.rowid, a.offset, a.date, a.vwretd, d.ret
    from work.evtdays0 as a left join crsp.dsf as d
      on a.permno=d.permno and a.date=d.date;
    create table work.car as
    select rowid, count(ret) as nret,
           sum(case when not missing(ret) and not missing(vwretd)
                    then ret-vwretd else . end) as car
    from work.evtdays group by rowid;
quit;

proc sql;
    create table work.base0 as
    select a.*, b.lagraw, b.lagdate, b.lagpends, b.laggap, b.nlagbest,
           c.permno, c.siccd, c.shrcd, c.nperm, c.nsic,
           p.pdate, p.price, p.cfac_p,
           e.eidx, e.evtrade,
           f.forecast, f.fdate, f.analyst_n,
           g.cfac_a, g.cfac_l, g.cfac_f,
           g.fdate_a, g.fdate_l, g.fdate_f,
           h.nret, h.car
    from work.qcurr as a
    left join work.lagagg as b on a.rowid=b.rowid
    left join work.linkagg as c on a.rowid=c.rowid
    left join work.pdata as p on a.rowid=p.rowid
    left join work.eventdate as e on a.rowid=e.rowid
    left join work.fcast as f on a.rowid=f.rowid
    left join work.factors as g on a.rowid=g.rowid
    left join work.car as h on a.rowid=h.rowid;
quit;

data work.base;
    set work.base0;
    if nlagbest ne 1 then lagraw=.;
    if nperm ne 1 or nsic ne 1 then do; permno=.; siccd=.; end;
    if not missing(cfac_p) and cfac_p>0 and not missing(cfac_a) and cfac_a>0 then
        eps_t= value * cfac_p / cfac_a;
    if not missing(cfac_p) and cfac_p>0 and not missing(cfac_l) and cfac_l>0 then
        eps_lag= lagraw * cfac_p / cfac_l;
    if not missing(cfac_p) and cfac_p>0 and not missing(cfac_f) and cfac_f>0 then
        eps_forecast= forecast * cfac_p / cfac_f;
    if not missing(price) and price>0 then do;
        loss=eps_t/price;
        earnchange=(eps_t-eps_lag)/price;
        surprise=(eps_t-eps_forecast)/price;
    end;
    if not missing(pdate) and not missing(qbegin) then price_age_days=pdate-qbegin;
    loss_ok=not missing(loss);
    change_ok=not missing(earnchange);
    surprise_ok=not missing(surprise);
    price_ok=not missing(price) and price>0 and not missing(cfac_p) and cfac_p>0;
    lag_ok=nlagbest=1 and not missing(lagraw);
    forecast_ok=not missing(forecast) and not missing(fdate) and not missing(cfac_f);
    link_ok=nperm=1 and nsic=1 and not missing(permno) and not missing(siccd);
    car_ok=nret=3 and not missing(car);
    industry_excluded=(siccd>=4400 and siccd<=5000) or (siccd>=6000 and siccd<=6500);
    format qbegin lagtarget pends anndats lagpends lagdate fdate pdate evtrade fdate_a fdate_l fdate_f date9.;
run;

/* Diagnostic only: do not screen observations on the elapsed price age. */
title 'First available CRSP price age from fiscal-quarter start, all current actuals';
proc freq data=work.base;
    tables price_age_days / missing;
run;
proc means data=work.base n nmiss mean p50 p90 p95 p99 max maxdec=2;
    var price_age_days;
run;

/* Staged construction and missing-versus-zero diagnostics. */
proc sql;
    create table work.attrition as
    select count(*) as n_current_actuals,
           sum(lag_ok) as n_with_t4_actual,
           sum(forecast_ok) as n_with_timely_forecast,
           sum(price_ok) as n_with_begin_price,
           sum(link_ok) as n_with_unique_link_sic,
           sum(car_ok) as n_with_complete_car,
           sum(loss_ok and change_ok and surprise_ok) as n_with_all_threshold_inputs,
           sum(not missing(value) and value=0) as n_zero_current_eps,
           sum(missing(value)) as n_missing_current_eps,
           sum(not missing(forecast) and forecast=0) as n_zero_forecast,
           sum(missing(forecast)) as n_missing_forecast,
           sum(missing(car)) as n_missing_car
    from work.base;
    create table work.sicstage as
    select count(*) as n_pre_sic,
           sum(missing(siccd)) as n_missing_sic,
           sum(industry_excluded) as n_sic_excluded
    from work.base;
quit;
title 'Full 1985–2002 firm-quarter construction attrition';
proc print data=work.attrition noobs; run;
proc print data=work.sicstage noobs; run;

data work.precar;
    set work.base;
    if not lag_ok or not forecast_ok or not price_ok or not link_ok or not car_ok then delete;
    if not loss_ok or not change_ok or not surprise_ok then delete;
    if industry_excluded then delete;
run;
proc univariate data=work.precar noprint;
    var car;
    output out=work.carpct pctlpts=1 99 pctlpre=carp_;
run;
data _null_;
    set work.carpct;
    call symputx('carp1',carp_1);
    call symputx('carp99',carp_99);
run;
data work.mainfq;
    set work.precar;
    if car < &carp1 or car > &carp99 then delete;
run;

title 'Final main firm-quarter sample and input distributions';
proc sql;
    select count(*) as final_firm_quarters,
           min(pends) as first_period_end format=date9.,
           max(pends) as last_period_end format=date9.,
           min(anndats) as first_announcement format=date9.,
           max(anndats) as last_announcement format=date9.,
           min(pdate) as first_price_date format=date9.,
           max(pdate) as last_price_date format=date9.
    from work.mainfq;
    select count(*) as main_rows_before_car_trim,
           sum(car < &carp1) as low_car_removed,
           sum(car > &carp99) as high_car_removed,
           &carp1 as car_p1, &carp99 as car_p99
    from work.precar;
quit;
proc means data=work.mainfq n nmiss mean std min p1 p50 p99 max maxdec=6;
    var price price_age_days eps_t eps_lag eps_forecast loss earnchange surprise car;
run;
proc freq data=work.mainfq;
    tables qyear price_age_days / missing;
run;

/* Preserve the constructed common firm-quarter sample under scratch for audit. */
data aud.main_firmquarter;
    set work.mainfq(keep=rowid ticker cusip permno pends qyear qbegin pdate
        anndats evtrade lagpends lagdate laggap fdate siccd price price_age_days cfac_p cfac_a cfac_l cfac_f
        value lagraw forecast eps_t eps_lag eps_forecast loss earnchange surprise car analyst_n);
run;

/* Common-sample threshold distributions; zero maps to bin 0, right of zero. */
data work.binned;
    set work.mainfq;
    loss_bin=floor(loss/0.0025);
    change_bin=floor(earnchange/0.0025);
    surprise_bin=floor(surprise/0.0025);
run;
proc sql;
    create table work.annual0 as
    select qyear as year, count(*) as n_firmquarter,
       sum(loss_bin=-2) as loss_m2,
       sum(loss_bin=-1) as loss_m1,
       sum(loss_bin=0) as loss_0,
       sum(change_bin=-2) as change_m2,
       sum(change_bin=-1) as change_m1,
       sum(change_bin=0) as change_0,
       sum(surprise_bin=-2) as surprise_m2,
       sum(surprise_bin=-1) as surprise_m1,
       sum(surprise_bin=0) as surprise_0
    from work.binned group by qyear;
quit;
data work.annual;
    set work.annual0;
    nstd=1000;
    pl2=loss_m2/n_firmquarter; pl1=loss_m1/n_firmquarter; p0=loss_0/n_firmquarter;
    loss_var=1000*pl1*(1-pl1)+250*(pl2+p0)*(1-pl2-p0);
    z_loss=(1000*pl1-500*(pl2+p0))/sqrt(loss_var);
    pl2=change_m2/n_firmquarter; pl1=change_m1/n_firmquarter; p0=change_0/n_firmquarter;
    change_var=1000*pl1*(1-pl1)+250*(pl2+p0)*(1-pl2-p0);
    z_decrease=(1000*pl1-500*(pl2+p0))/sqrt(change_var);
    pl2=surprise_m2/n_firmquarter; pl1=surprise_m1/n_firmquarter; p0=surprise_0/n_firmquarter;
    surprise_var=1000*pl1*(1-pl1)+250*(pl2+p0)*(1-pl2-p0);
    z_surprise=(1000*pl1-500*(pl2+p0))/sqrt(surprise_var);
    keep year n_firmquarter z_loss z_decrease z_surprise;
run;
proc sort data=work.annual; by year; run;
title 'Annual standardized differences for assigned trend fits';
proc means data=work.annual n nmiss mean std min max maxdec=6;
    var n_firmquarter z_loss z_decrease z_surprise;
run;
proc sql;
    select count(*) as annual_observations,
           min(year) as first_year, max(year) as last_year,
           sum(n_firmquarter) as total_firm_quarters
    from work.annual;
quit;

data out.main;
    set work.annual;
run;
